Data Centric AI: model explanation using Influence Analysis

In [1]:
from ansi_colors import *
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split 

import tensorflow as tf
tf.config.run_functions_eagerly(True)
print(tf.executing_eagerly())

from keras.losses import BinaryCrossentropy 
from keras.utils import to_categorical 

from keras import Sequential, layers

True


In [2]:
dataset = '../dataset/titanic.csv'
df = pd.read_csv(dataset)
original_df = df.copy()
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


'Name' and 'Ticket' have too much unique value, furthermore 'Cabin' has too much null numbers. We also remove that here.

In [3]:
df.rename(columns={'PassengerId': 'ID'}, inplace=True)
df.drop(columns=['Name', 'Ticket', 'Cabin'], inplace=True)
df

,ID,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.2500,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.9250,S
3,4,1,1,female,35.0,1,0,53.1000,S
4,5,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,13.0000,S
887,888,1,1,female,19.0,0,0,30.0000,S
888,889,0,3,female,NaN,1,2,23.4500,S
889,890,1,1,male,26.0,0,0,30.0000,C


Now check if there are any NA values within the dataframe.

In [4]:
print(df.isna().sum())

ID            0
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64


'Embarked' has only 2 NA values, we can remove these 2 rows for now.

As for age, there are 177 missing values, so it's not feasible to drop the rows in this case because too much data would be lost. One possible approach is to set the missing values to zero and add a column (with a value of 0 or 1) to indicate whether the value is missing

In [5]:
df.dropna(subset=['Embarked'], inplace=True)
df = df.copy()
df['MissAge'] = df['Age'].isna().astype(int)
df.fillna({'Age': 0}, inplace=True)
print(df.isna().sum())
df

ID          0
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
MissAge     0
dtype: int64


,ID,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MissAge
0,1,0,3,male,22.0,1,0,7.2500,S,0
1,2,1,1,female,38.0,1,0,71.2833,C,0
2,3,1,3,female,26.0,0,0,7.9250,S,0
3,4,1,1,female,35.0,1,0,53.1000,S,0
4,5,0,3,male,35.0,0,0,8.0500,S,0
...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,13.0000,S,0
887,888,1,1,female,19.0,0,0,30.0000,S,0
888,889,0,3,female,0.0,1,2,23.4500,S,1
889,890,1,1,male,26.0,0,0,30.0000,C,0


Now let's transform the text features into numerical features.

In [6]:
sex_label_enc = LabelEncoder()
df['Sex'] = sex_label_enc.fit_transform(df['Sex'])

emb_label_enc = LabelEncoder()
df['Embarked'] = emb_label_enc.fit_transform(df['Embarked'])

df

,ID,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MissAge
0,1,0,3,1,22.0,1,0,7.2500,2,0
1,2,1,1,0,38.0,1,0,71.2833,0,0
2,3,1,3,0,26.0,0,0,7.9250,2,0
3,4,1,1,0,35.0,1,0,53.1000,2,0
4,5,0,3,1,35.0,0,0,8.0500,2,0
...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,1,27.0,0,0,13.0000,2,0
887,888,1,1,0,19.0,0,0,30.0000,2,0
888,889,0,3,0,0.0,1,2,23.4500,2,1
889,890,1,1,1,26.0,0,0,30.0000,0,0


Normalize the data

In [7]:
norm = StandardScaler()
col = ['Age', 'Fare']

df[col] = norm.fit_transform(df[col])
df

,ID,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MissAge
0,1,0,3,1,-0.099150,1,0,-0.500240,2,0
1,2,1,1,0,0.812389,1,0,0.788947,0,0
2,3,1,3,0,0.128735,0,0,-0.486650,2,0
3,4,1,1,0,0.641476,1,0,0.422861,2,0
4,5,0,3,1,0.641476,0,0,-0.484133,2,0
...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,1,0.185706,0,0,-0.384475,2,0
887,888,1,1,0,-0.270063,0,0,-0.042213,2,0
888,889,0,3,0,-1.352516,1,2,-0.174084,2,1
889,890,1,1,1,0.128735,0,0,-0.042213,0,0


Now we're moving towards training the model.

In [8]:
X = df.drop(columns=['Survived'])
y = df['Survived']

IDs = X['ID'].values.reshape(-1,1).astype(np.float32)
IDs = IDs / 1e7

X = X.drop(columns=['ID']).values.astype(np.float32)
X = np.hstack((X, IDs))

y = to_categorical(y.values, num_classes=2)

print(X.shape)
print(y.shape)

(889, 9)
(889, 2)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(711, 9)
(178, 9)
(711, 2)
(178, 2)


In [10]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
print(len(train_ds))
print(len(test_ds))

711
178


Now we can build and train a simple NN and then we'll try to explain model.

In [11]:
model = Sequential([
    layers.Dense(32, activation='relu', input_shape=(9,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(2, activation='sigmoid')
])

#model.compile(optimizer='adam', loss=BinaryCrossentropy(from_logits=True), metrics=['accuracy'])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds.batch(32), epochs=10, validation_data=test_ds.batch(32), verbose=2)
loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(f"--- {CYAN}Performance{RESET} ---")
print(f"{GREEN}Loss: {loss}")
print(f"Accuracy: {acc}{RESET}")

Epoch 1/10
23/23 - 0s - loss: 0.6550 - accuracy: 0.6203 - val_loss: 0.6430 - val_accuracy: 0.6124 - 226ms/epoch - 10ms/step
Epoch 2/10
23/23 - 0s - loss: 0.6336 - accuracy: 0.6188 - val_loss: 0.6270 - val_accuracy: 0.6124 - 196ms/epoch - 9ms/step
Epoch 3/10
23/23 - 0s - loss: 0.6190 - accuracy: 0.6188 - val_loss: 0.6119 - val_accuracy: 0.6124 - 187ms/epoch - 8ms/step
Epoch 4/10
23/23 - 0s - loss: 0.6048 - accuracy: 0.6287 - val_loss: 0.5963 - val_accuracy: 0.6236 - 187ms/epoch - 8ms/step
Epoch 5/10
23/23 - 0s - loss: 0.5900 - accuracy: 0.6540 - val_loss: 0.5793 - val_accuracy: 0.6742 - 187ms/epoch - 8ms/step
Epoch 6/10
23/23 - 0s - loss: 0.5740 - accuracy: 0.6695 - val_loss: 0.5608 - val_accuracy: 0.6966 - 189ms/epoch - 8ms/step
Epoch 7/10
23/23 - 0s - loss: 0.5572 - accuracy: 0.7089 - val_loss: 0.5423 - val_accuracy: 0.7416 - 194ms/epoch - 8ms/step
Epoch 8/10
23/23 - 0s - loss: 0.5406 - accuracy: 0.7370 - val_loss: 0.5254 - val_accuracy: 0.7809 - 193ms/epoch - 8ms/step
Epoch 9/10
23/2

Now we have the result from the model, we can test it with influenciae now.

In [14]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

import warnings

In [17]:
warnings.filterwarnings('ignore')
unreduced_loss = BinaryCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
influence_model = InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss)

# First order influence calculator
ihvp_calculator = ExactIHVP(influence_model, train_dataset=train_ds.shuffle(100).batch(4))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds.batch(8), ihvp_calculator)


samples_to_explain = test_ds.take(5).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(8), k=3, order=ORDER.DESCENDING)

# Print influential samples
for (sample, label), top_k_values, top_k_samples in explanation_ds.as_numpy_iterator():
    #Remember to use round instead of int :<
    sample_id = round(sample[0][-1] * 1e7)
    sample_original = original_df[original_df['PassengerId'] == sample_id]
    print(f"\n{GREEN}Test Sample ID: {sample_id}{RESET}")
    print(f"{CYAN}Original sample from DataFrame:{RESET}")
    print(sample_original[['Survived']])
    influential_ids = [round(s[-1] * 1e7) for s in top_k_samples[0]]
    for i, (inf_id, score) in enumerate(zip(influential_ids, top_k_values[0])):
        inf_sample_original = original_df[original_df['PassengerId'] == inf_id]
        print(F"- Influential Sample {i} -> ID: {inf_id}, Influence Score: {score}")
        print(inf_sample_original[['Survived']])


Test Sample ID: 282
Original sample from DataFrame:
     Survived
281         0
- Influential Sample 0 -> ID: 551, Influence Score: 2.630390167236328
     Survived
550         1
- Influential Sample 1 -> ID: 446, Influence Score: 2.451584577560425
     Survived
445         1
- Influential Sample 2 -> ID: 789, Influence Score: 1.8324886560440063
     Survived
788         1

Test Sample ID: 436
Original sample from DataFrame:
     Survived
435         1
- Influential Sample 0 -> ID: 446, Influence Score: 25.435028076171875
     Survived
445         1
- Influential Sample 1 -> ID: 551, Influence Score: 19.976882934570312
     Survived
550         1
- Influential Sample 2 -> ID: 616, Influence Score: 17.10871124267578
     Survived
615         1

Test Sample ID: 40
Original sample from DataFrame:
    Survived
39         1
- Influential Sample 0 -> ID: 804, Influence Score: 22.327003479003906
     Survived
803         1
- Influential Sample 1 -> ID: 645, Influence Score: 20.477008819580078